[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kevin-innovation/jupyter-lecture/blob/main/python-web-automation/lectures/01/%5B%ED%95%99%EC%83%9D%EC%9A%A9%5D%20%EB%A0%88%EC%8A%A8%2001%20%E2%80%94%20%EC%9B%B9%20%EC%9E%90%EB%8F%99%ED%99%94%20%EA%B0%9C%EC%9A%94%20%2B%20HTTP%20%2B%20BeautifulSoup%20%EC%9E%85%EB%AC%B8.ipynb)

# 레슨 01 — 웹 자동화 개요 + HTTP + BeautifulSoup 입문

In [ ]:
import os
import re
import time
import csv
from pathlib import Path
from urllib.parse import urljoin

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/01/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', text))

print('colab:', IS_COLAB)
print('data base:', DATA_BASE)


---

## 4. HTML 파일 읽기

첫 샘플은 mini_shop.html이다. 실제 쇼핑몰이 아니라 수업용으로 만든 합성 HTML이다.

In [ ]:
html = load_text('mini_shop.html')
print(type(html))
print(len(html))
print(html[:300])


HTML은 그냥 긴 문자열이다. 문자열 상태에서는 태그를 찾기 어렵기 때문에 BeautifulSoup 객체로 바꾼다.

In [ ]:
soup = BeautifulSoup(html, 'html.parser')
print(soup.title.text.strip())
print(soup.select_one('h1').text.strip())


---

## 5. CSS selector로 원하는 요소 고르기

웹 자동화에서 가장 많이 쓰는 문법은 CSS selector다.

- 'h1': h1 태그 선택
- '.product-card': class가 product-card인 요소 선택
- '#main': id가 main인 요소 선택
- 'article.product-card': article 태그이면서 product-card 클래스인 요소 선택
- '.product-card .name': product-card 안쪽의 name 클래스 선택

In [ ]:
cards = soup.select('.product-card')
print('상품 카드 수:', len(cards))

first = cards[0]
print(first.select_one('.name').text.strip())
print(first.select_one('.price').text.strip())
print(first['data-category'])


---

## 6. 텍스트 정리와 숫자 변환

웹에서 가져온 값은 대부분 문자열이다. 가격처럼 쉼표와 원 문자가 섞인 문자열은 숫자로 바꿔야 비교와 정렬이 가능하다.

In [ ]:
price_text = first.select_one('.price').text.strip()
price = clean_int(price_text)
print(price_text, '->', price)


---

## 7. 여러 상품을 표 형태로 만들기

반복문으로 각 상품 카드에서 같은 위치의 값을 꺼낸다. 리스트 안에 딕셔너리를 쌓으면 표처럼 다루기 쉽다.

In [ ]:
products = []
for card in cards:
    item = {
        'name': card.select_one('.name').text.strip(),
        'category': card['data-category'],
        'price': clean_int(card.select_one('.price').text),
        'rating': float(card.select_one('.rating').text.replace('★', '').strip()),
        'stock': clean_int(card.select_one('.stock').text),
        'detail_url': urljoin('https://example.com', card.select_one('a.detail')['href']),
    }
    products.append(item)

print(products[0])
print('총 상품:', len(products))


---

## 8. 조건으로 걸러 보기

리스트 컴프리헨션을 쓰면 원하는 조건의 데이터만 뽑을 수 있다.

In [ ]:
expensive = [p for p in products if p['price'] >= 1500000]
high_rating = [p for p in products if p['rating'] >= 4.7]
out_of_stock = [p for p in products if p['stock'] == 0]

print('150만원 이상:', len(expensive))
print('평점 4.7 이상:', len(high_rating))
print('품절:', len(out_of_stock))


---

## 9. 공지 페이지 파싱

두 번째 샘플은 notices.html이다. 공지 목록처럼 행이 반복되는 구조를 연습한다.

In [ ]:
notice_html = load_text('notices.html')
notice_soup = BeautifulSoup(notice_html, 'html.parser')
rows = notice_soup.select('li.notice-item')
print('공지 수:', len(rows))

first_notice = rows[0]
print(first_notice.select_one('.date').text.strip())
print(first_notice.select_one('.title').text.strip())
print(first_notice.select_one('.department').text.strip())


---

## 10. CSV로 저장하기

수집 결과는 화면에 print만 하지 말고 파일로 남겨야 한다. 이번 레슨에서는 csv 모듈을 쓴다.

In [ ]:
output_path = 'lesson01_products.csv'
with open(output_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['name', 'category', 'price', 'rating', 'stock', 'detail_url'])
    writer.writeheader()
    writer.writerows(products)

print('saved:', output_path)


---

## 11. 현업 활용 사례

웹 자동화는 데이터 분석보다 먼저 쓰이는 경우가 많다. 분석할 데이터가 파일로 주어지지 않을 때, 웹 페이지의 공지·가격·리뷰·채용공고·자료실 링크를 정리해 데이터셋을 만든다.

예를 들어 가격 비교 서비스는 상품명, 가격, 재고, 판매처 링크를 주기적으로 확인한다. 다만 실서비스는 단순 requests만 쓰지 않는다. robots.txt, 약관, 요청 제한, IP 차단 방지, 중복 저장 방지, 변경 이력 관리까지 운영 규칙이 필요하다.

1강에서는 운영 전체가 아니라 '한 페이지를 안전하게 읽고 표로 바꾸는 기본기'만 익힌다.

## 12. HTTP 응답을 안전하게 확인하는 법

웹 자동화에서 가장 흔한 실수는 "HTML이 왔겠지"라고 가정하고 바로 파싱하는 것이다. 실제 사이트에서는 URL 오타, 권한 없음, 서버 오류, 너무 빠른 요청 때문에 HTML 대신 에러 페이지가 올 수 있다. 그래서 요청 결과를 파싱하기 전에 응답 상태를 확인해야 한다.

> **🛡️ 웹 자동화 안전 한스푼 — HTTP 요청/응답과 status_code**
>
> - **뜻**: 요청은 클라이언트가 서버에 보내는 질문이고, 응답은 서버가 돌려주는 결과다. `status_code` 는 그 결과의 상태 번호다.
> - **왜 중요한가**: 200이 아니면 원하는 HTML이 아닐 수 있다. 404는 주소 없음, 403은 거부, 429는 너무 많은 요청을 뜻할 수 있다.
> - **수업 기준**: 외부 요청 코드는 항상 `timeout` 과 상태 확인을 같이 둔다.
> - **실수 예시**: `requests.get(url).text` 만 쓰고 상태를 확인하지 않은 채 BeautifulSoup에 넘긴다.

In [ ]:
sample_url = DATA_BASE + '/mini_shop.html' if DATA_BASE.startswith('http') else None

if sample_url:
    response = requests.get(sample_url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
    print('status:', response.status_code)
    response.raise_for_status()
    print(response.text[:80])
else:
    print('로컬 실행 중이므로 외부 HTTP 요청 예시는 건너뜁니다.')


> **🛡️ 웹 자동화 안전 한스푼 — timeout**
>
> - **뜻**: 서버가 일정 시간 안에 응답하지 않으면 기다리기를 멈추는 제한 시간이다.
> - **왜 중요한가**: timeout이 없으면 네트워크 문제 하나로 노트북 셀이 계속 멈춰 있을 수 있다.
> - **수업 기준**: `requests.get(url, timeout=10)` 을 기본으로 쓴다.
> - **실수 예시**: 여러 URL을 반복 요청하면서 timeout을 넣지 않아 수업 전체가 멈춘다.

`raise_for_status()` 는 400번대/500번대 응답을 그냥 넘어가지 않게 해 준다. 처음에는 에러가 뜨는 것이 불편해 보이지만, 조용히 틀린 HTML을 파싱하는 것보다 훨씬 안전하다.

---

## 13. selector 디버깅 루틴

selector가 틀리면 BeautifulSoup은 에러를 내지 않고 빈 리스트를 돌려주는 경우가 많다. 그래서 한 번에 긴 코드를 쓰지 말고 다음 순서로 확인한다.

1. 문서 전체가 제대로 읽혔는지 `len(html)` 을 본다.
2. `soup.title` 또는 `h1` 처럼 확실한 요소를 먼저 찾는다.
3. 반복 단위 selector의 개수를 `len(soup.select(...))` 로 본다.
4. 첫 번째 반복 단위 하나에서 내부 요소를 확인한다.
5. 그 다음에야 반복문으로 전체를 처리한다.

In [ ]:
debug_cards = soup.select('.product-card')
print('cards:', len(debug_cards))

if debug_cards:
    sample = debug_cards[0]
    for selector in ['.name', '.price', '.rating', '.stock', 'a.detail']:
        found = sample.select_one(selector)
        print(selector, '=>', found.text.strip() if found else '없음')


> **🛡️ 웹 자동화 안전 한스푼 — HTML selector**
>
> - **뜻**: HTML에서 원하는 태그를 고르는 주소 같은 표현이다. `.class`, `#id`, `tag[attr=value]` 를 조합한다.
> - **왜 중요한가**: selector가 불안정하면 사이트 구조가 조금만 바뀌어도 자동화가 깨진다.
> - **수업 기준**: 반복 단위 selector를 먼저 고르고, 내부 selector는 그 안에서 다시 찾는다.
> - **실수 예시**: 전체 문서에서 `.price` 를 한 번만 찾아 첫 상품 가격만 계속 저장한다.

selector가 비어 있으면 HTML을 다시 보는 것이 먼저다. 코드를 더 복잡하게 만들기 전에 "내가 고르려는 반복 단위가 실제로 어떤 태그와 class를 갖는가"를 눈으로 확인한다.

---

## 14. 상대 URL과 절대 URL

HTML 안의 링크는 `/products/air-13` 처럼 상대 경로로 들어 있는 경우가 많다. 사람이 브라우저로 볼 때는 현재 사이트 주소가 자동으로 붙지만, CSV로 저장할 때는 기준 주소를 붙여 절대 URL로 바꿔야 한다.

In [ ]:
relative_href = first.select_one('a.detail')['href']
absolute_href = urljoin('https://example.com/shop/', relative_href)
print(relative_href)
print(absolute_href)


상대 URL을 그대로 저장하면 나중에 CSV만 열었을 때 링크가 어디를 가리키는지 알기 어렵다. `urljoin` 은 기준 주소와 상대 경로를 안전하게 합쳐 준다.

> **🛡️ 웹 자동화 안전 한스푼 — relative URL / absolute URL**
>
> - **뜻**: 상대 URL은 현재 사이트를 기준으로 한 짧은 주소이고, 절대 URL은 `https://...` 로 시작하는 완전한 주소다.
> - **왜 중요한가**: 수집 결과를 CSV로 저장하면 기준 사이트 정보가 사라질 수 있다.
> - **수업 기준**: 링크를 저장할 때는 `urljoin` 으로 절대 URL 형태를 만든다.
> - **실수 예시**: `/notice/12` 만 저장해서 나중에 어느 사이트의 공지인지 알 수 없다.

---

## 15. robots.txt와 요청 간격

이번 레슨은 합성 HTML 파일을 쓰기 때문에 실제 서버에 반복 요청하지 않는다. 그래도 처음부터 robots.txt와 요청 간격을 말하는 이유는 습관 때문이다. 자동화 코드는 반복문을 쓰는 순간 사람이 클릭하는 속도보다 훨씬 빨라질 수 있다.

> **🛡️ 웹 자동화 안전 한스푼 — robots.txt**
>
> - **뜻**: 사이트가 자동화 프로그램에게 어느 경로를 허용하거나 제한하는지 알려주는 참고 파일이다.
> - **왜 중요한가**: 공개 페이지라도 운영자가 자동 접근을 제한하고 싶을 수 있다.
> - **수업 기준**: 실제 사이트 예제는 robots와 약관을 확인한 뒤 강사가 지정한 범위에서만 요청한다.
> - **실수 예시**: "브라우저에서 보이니까 마음대로 반복 요청해도 된다"고 생각한다.

> **🛡️ 웹 자동화 안전 한스푼 — 요청 간격(rate limit)**
>
> - **뜻**: 여러 페이지를 요청할 때 요청 사이에 쉬는 시간을 두는 규칙이다.
> - **왜 중요한가**: 너무 빠른 반복 요청은 서버 부하나 차단의 원인이 된다.
> - **수업 기준**: 실제 외부 요청 반복문에는 최소 `time.sleep(1)` 을 둔다.
> - **실수 예시**: 1초에 수십 번씩 URL을 바꿔 요청한다.

In [ ]:
targets_text = load_text('targets.csv')
print(targets_text.splitlines()[:4])

# 실제 사이트 요청 예시는 아니다. 반복 처리 구조만 보여준다.
for index, filename in enumerate(['mini_shop.html', 'notices.html'], start=1):
    text = load_text(filename)
    print(index, filename, len(text))
    time.sleep(0.2)  # 수업 fixture라 짧게 둔다. 실제 외부 사이트는 더 길게 둔다.


---

## 16. 실무에서는 무엇을 더 추가하나

실무 자동화는 1강 코드보다 더 많은 보호 장치를 둔다.

| 항목 | 1강에서 하는 것 | 실무에서 추가하는 것 |
|---|---|---|
| 요청 | fixture 또는 raw 파일 읽기 | 재시도, 요청 간격, 실패 로그 |
| 파싱 | selector로 텍스트 추출 | selector 변경 감지, 누락 필드 알림 |
| 저장 | CSV 1개 저장 | 날짜별 파일명, 중복 제거, DB 저장 |
| 검증 | 개수와 일부 값 출력 | 스키마 검증, null 비율, 이전 결과와 비교 |
| 운영 | 노트북 수동 실행 | 스케줄러, 알림, 장애 대응 |

이 표를 보면 1강의 목적이 분명해진다. 오늘은 실무 전체를 다 만들지 않는다. 대신 반복 단위를 고르고, 내부 값을 읽고, 숫자로 바꾸고, CSV로 저장하는 기초 루틴을 정확히 익힌다. 이 루틴이 흔들리면 페이지네이션, 브라우저 자동화, 재시도 같은 뒤 레슨도 모두 흔들린다.

---

## 17. 제출 전 마무리 체크

이번 레슨을 마치기 전에 아래 질문에 답해 본다.

- HTML 문자열을 BeautifulSoup 객체로 바꾸는 이유를 말할 수 있는가?
- `select` 와 `select_one` 의 차이를 말할 수 있는가?
- class selector 앞에 `.` 을 붙이는 이유를 설명할 수 있는가?
- 가격/조회수 문자열을 숫자로 바꾸지 않으면 어떤 문제가 생기는가?
- 상대 URL을 절대 URL로 바꾸는 이유를 말할 수 있는가?
- 실제 사이트에서 같은 코드를 실행하기 전에 robots.txt, 약관, 요청 간격을 확인해야 하는 이유를 말할 수 있는가?

## 데이터 출처와 안전 규칙

이 레슨의 `mini_shop.html`, `notices.html`, `targets.csv`, `robots_sample.txt` 는 수업용 합성 데이터다. 실존 사이트나 개인 정보를 포함하지 않는다. `requests` 예제는 raw GitHub의 수업 파일 또는 로컬 fixture만 읽도록 설계되어 있다. 학생은 강사가 별도로 허가하지 않은 실제 사이트에 반복 요청을 보내지 않는다.

# 레슨 01 — 실습 문제

lecture.ipynb를 본 뒤 진행하는 학생용 문제 노트북이다. 이번 레슨은 정적 HTML 파일을 읽고, BeautifulSoup selector로 필요한 값을 뽑아 표 형태로 정리하는 것이 목표다.

## 통과 기준

- 총 15문제 중 12문제 이상 정상 출력이면 통과.
- 문제 1~5: 환경, HTML 기본 구조, 단일 요소 추출.
- 문제 6~10: 상품 카드 반복 추출과 조건 필터.
- 문제 11~14: 공지/대상 CSV 파싱.
- 문제 15: 수집 결과를 근거로 한 문장 요약 작성.

## 0. 환경 셀

In [ ]:
import os
import re
import time
import csv
from pathlib import Path
from urllib.parse import urljoin

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/01/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', text))

print('colab:', IS_COLAB)
print('data base:', DATA_BASE)


---

## 문제 1 — 환경 셀 실행과 파일 목록 확인

mini_shop.html, notices.html, targets.csv를 load_text로 읽고 각 문자열 길이를 출력한다.

**기대 결과 형태**: 파일명과 글자 수가 3줄로 나온다.

**빈칸 힌트**: 코드 셀의 빈칸을 채운다. selector 문자열과 딕셔너리 key는 lecture 노트북의 예제와 HTML 구조를 같이 보고 결정한다.

In [ ]:
# TODO: ____ 부분을 직접 채우세요.
shop_html = ____(____)
notice_html = ____(____)
target_csv = ____(____)
print('shop:', ____)
print('notice:', ____)
print('targets:', ____)


---

## 문제 2 — HTML 제목과 h1 찾기

mini_shop.html에서 title과 h1 텍스트를 출력한다.

**기대 결과 형태**: 문서 제목 1줄, 화면 제목 1줄이 나온다.

**빈칸 힌트**: 코드 셀의 빈칸을 채운다. selector 문자열과 딕셔너리 key는 lecture 노트북의 예제와 HTML 구조를 같이 보고 결정한다.

In [ ]:
# TODO: ____ 부분을 직접 채우세요.
soup = ____(shop_html, 'html.parser')
page_title = soup.____.text.strip()
main_title = soup.____('____').text.strip()
print(page_title)
print(main_title)


---

## 문제 3 — 상품 카드 개수 세기

class가 product-card인 요소를 모두 선택해 개수를 출력한다.

**기대 결과 형태**: 상품 카드 수가 한 줄로 나온다.

**빈칸 힌트**: 코드 셀의 빈칸을 채운다. selector 문자열과 딕셔너리 key는 lecture 노트북의 예제와 HTML 구조를 같이 보고 결정한다.

In [ ]:
# TODO: ____ 부분을 직접 채우세요.
cards = soup.____('____')
print('상품 카드 수:', ____)


---

## 문제 4 — 첫 번째 상품 정보 읽기

첫 상품의 이름, 카테고리, 가격 문자열, 상세 링크를 출력한다.

**기대 결과 형태**: 이름/category/가격/href가 각각 나온다.

**빈칸 힌트**: 코드 셀의 빈칸을 채운다. selector 문자열과 딕셔너리 key는 lecture 노트북의 예제와 HTML 구조를 같이 보고 결정한다.

In [ ]:
# TODO: ____ 부분을 직접 채우세요.
first = cards[____]
name = first.____('____').text.strip()
category = first['____']
price_text = first.____('____').text.strip()
href = first.____('____')['____']
print(name, category, price_text, href)


---

## 문제 5 — 가격 문자열을 숫자로 바꾸기

첫 상품 가격에서 숫자만 남겨 int로 변환한다.

**기대 결과 형태**: 정수 가격이 출력된다.

**빈칸 힌트**: 코드 셀의 빈칸을 채운다. selector 문자열과 딕셔너리 key는 lecture 노트북의 예제와 HTML 구조를 같이 보고 결정한다.

In [ ]:
# TODO: ____ 부분을 직접 채우세요.
price_text = first.select_one('.price').text
price = ____(price_text)
print(price, type(price))


---

## 문제 6 — 모든 상품을 딕셔너리 리스트로 만들기

각 상품에서 name/category/price/rating/stock/detail_url을 추출해 products 리스트를 만든다.

**기대 결과 형태**: 첫 딕셔너리와 전체 개수가 출력된다.

**빈칸 힌트**: 코드 셀의 빈칸을 채운다. selector 문자열과 딕셔너리 key는 lecture 노트북의 예제와 HTML 구조를 같이 보고 결정한다.

In [ ]:
# TODO: ____ 부분을 직접 채우세요.
products = []
for card in cards:
    item = {
        'name': card.select_one('____').text.strip(),
        'category': card['____'],
        'price': ____(card.select_one('____').text),
        'rating': float(card.select_one('____').text.replace('★', '').strip()),
        'stock': ____(card.select_one('____').text),
        'detail_url': urljoin('https://example.com', card.select_one('____')['____']),
    }
    products.append(item)
print(products[0])
print(len(products))


---

## 문제 7 — 카테고리별 상품 수 세기

products에서 category별 상품 수를 딕셔너리로 집계한다.

**기대 결과 형태**: 카테고리 이름과 개수가 출력된다.

**빈칸 힌트**: 코드 셀의 빈칸을 채운다. selector 문자열과 딕셔너리 key는 lecture 노트북의 예제와 HTML 구조를 같이 보고 결정한다.

In [ ]:
# TODO: ____ 부분을 직접 채우세요.
counts = {}
for item in products:
    key = item['____']
    counts[key] = counts.get(key, 0) + ____
print(counts)


---

## 문제 8 — 150만원 이상 상품 찾기

가격이 1,500,000원 이상인 상품명과 가격을 출력한다.

**기대 결과 형태**: 조건을 만족하는 상품 목록이 나온다.

**빈칸 힌트**: 코드 셀의 빈칸을 채운다. selector 문자열과 딕셔너리 key는 lecture 노트북의 예제와 HTML 구조를 같이 보고 결정한다.

In [ ]:
# TODO: ____ 부분을 직접 채우세요.
expensive = [p for p in products if p['____'] >= ____]
for p in expensive:
    print(p['____'], p['____'])


---

## 문제 9 — 평점 4.7 이상 상품 찾기

rating이 4.7 이상인 상품만 골라 이름, 평점, 재고를 출력한다.

**기대 결과 형태**: 고평점 상품 목록이 나온다.

**빈칸 힌트**: 코드 셀의 빈칸을 채운다. selector 문자열과 딕셔너리 key는 lecture 노트북의 예제와 HTML 구조를 같이 보고 결정한다.

In [ ]:
# TODO: ____ 부분을 직접 채우세요.
top_rated = [p for p in products if p['____'] >= ____]
for p in top_rated:
    print(p['____'], p['____'], p['____'])


---

## 문제 10 — 품절 상품 찾기

stock이 0인 상품의 이름과 상세 URL을 출력한다.

**기대 결과 형태**: 품절 상품 목록이 나온다.

**빈칸 힌트**: 코드 셀의 빈칸을 채운다. selector 문자열과 딕셔너리 key는 lecture 노트북의 예제와 HTML 구조를 같이 보고 결정한다.

In [ ]:
# TODO: ____ 부분을 직접 채우세요.
sold_out = [p for p in products if p['____'] == ____]
for p in sold_out:
    print(p['____'], p['____'])


---

## 문제 11 — 공지 목록 파싱하기

notices.html에서 notice-item을 모두 골라 날짜, 제목, 부서를 추출한다.

**기대 결과 형태**: 첫 공지 딕셔너리와 전체 공지 수가 출력된다.

**빈칸 힌트**: 코드 셀의 빈칸을 채운다. selector 문자열과 딕셔너리 key는 lecture 노트북의 예제와 HTML 구조를 같이 보고 결정한다.

In [ ]:
# TODO: ____ 부분을 직접 채우세요.
notice_soup = ____(notice_html, 'html.parser')
notice_rows = notice_soup.____('____')
notices = []
for row in notice_rows:
    notices.append({
        'date': row.select_one('____').text.strip(),
        'title': row.select_one('____').text.strip(),
        'department': row.select_one('____').text.strip(),
        'views': ____(row.select_one('____').text),
    })
print(notices[0])
print(len(notices))


---

## 문제 12 — 운영팀 공지만 필터링하기

department가 운영팀인 공지만 골라 날짜와 제목을 출력한다.

**기대 결과 형태**: 운영팀 공지만 나온다.

**빈칸 힌트**: 코드 셀의 빈칸을 채운다. selector 문자열과 딕셔너리 key는 lecture 노트북의 예제와 HTML 구조를 같이 보고 결정한다.

In [ ]:
# TODO: ____ 부분을 직접 채우세요.
ops_notices = [n for n in notices if n['____'] == '____']
for n in ops_notices:
    print(n['____'], n['____'])


---

## 문제 13 — 조회수 상위 공지 3개 찾기

views 기준으로 notices를 내림차순 정렬해 상위 3개 제목과 조회수를 출력한다.

**기대 결과 형태**: 조회수 높은 순서로 3개가 나온다.

**빈칸 힌트**: 코드 셀의 빈칸을 채운다. selector 문자열과 딕셔너리 key는 lecture 노트북의 예제와 HTML 구조를 같이 보고 결정한다.

In [ ]:
# TODO: ____ 부분을 직접 채우세요.
top3 = sorted(notices, key=lambda n: n['____'], reverse=____)[:____]
for n in top3:
    print(n['____'], n['____'])


---

## 문제 14 — targets.csv에서 허용 대상만 읽기

targets.csv를 csv.DictReader로 읽고 allowed가 yes인 행만 출력한다.

**기대 결과 형태**: 허용 대상 파일 목록이 출력된다.

**빈칸 힌트**: 코드 셀의 빈칸을 채운다. selector 문자열과 딕셔너리 key는 lecture 노트북의 예제와 HTML 구조를 같이 보고 결정한다.

In [ ]:
# TODO: ____ 부분을 직접 채우세요.
import io
reader = csv.DictReader(io.StringIO(target_csv))
allowed = []
for row in reader:
    if row['____'] == '____':
        allowed.append(row)
for row in allowed:
    print(row['____'], row['____'])


---

## 문제 15 — 수집 요약 문장 만들기

상품/공지 데이터를 근거로 한 문장 요약 3개를 만든다.

**기대 결과 형태**: 상품 수, 품절 수, 조회수 상위 공지에 대한 문장이 출력된다.

**빈칸 힌트**: 코드 셀의 빈칸을 채운다. selector 문자열과 딕셔너리 key는 lecture 노트북의 예제와 HTML 구조를 같이 보고 결정한다.

In [ ]:
# TODO: ____ 부분을 직접 채우세요.
summary1 = f"총 상품 수는 {____}개입니다."
summary2 = f"품절 상품은 {____}개입니다."
summary3 = f"조회수 1위 공지는 '{____}'입니다."
print(summary1)
print(summary2)
print(summary3)


---

## 제출 전 셀프 체크

아래 항목을 스스로 확인한 뒤 제출한다. 이 표는 정답을 알려주기 위한 것이 아니라, 노트북이 위에서부터 다시 실행되어도 같은 결과가 나오는지 확인하기 위한 점검표다.

| 체크 항목 | 확인 방법 |
|---|---|
| 환경 셀을 먼저 실행했다 | `data base:` 출력이 보이는지 확인한다 |
| HTML 원문을 읽었다 | `len(shop_html)`, `len(notice_html)` 이 0보다 큰지 확인한다 |
| 반복 단위 개수를 먼저 확인했다 | 상품은 `.product-card`, 공지는 `li.notice-item` 개수를 출력한다 |
| selector 결과가 비어 있지 않다 | `select_one(...)` 결과가 `None` 이 아닌지 확인한다 |
| 숫자 변환을 했다 | 가격, 재고, 조회수는 `int` 또는 `float` 로 바뀌었는지 확인한다 |
| 링크를 절대 URL로 만들었다 | `detail_url` 이 `https://` 로 시작하는지 확인한다 |
| CSV 저장이 끝났다 | `lesson01_products.csv` 파일명이 출력되는지 확인한다 |

## 디버깅 규칙

문제가 막히면 긴 코드를 한 번에 고치지 않는다. 아래 순서대로 한 줄씩 확인한다.

1. HTML을 제대로 읽었는가?
2. BeautifulSoup 객체를 만들었는가?
3. 반복 단위 selector 개수가 맞는가?
4. 첫 번째 반복 단위에서 내부 selector가 잡히는가?
5. 문자열에서 숫자를 꺼내는 함수가 필요한가?
6. 반복문 안에서 만든 딕셔너리 key 이름이 뒤 문제와 같은가?

예를 들어 문제 8~10이 모두 실패하면 가격 조건식만 볼 것이 아니라 문제 6의 `products` 구조부터 다시 확인한다. `products[0]` 을 출력했을 때 `price`, `rating`, `stock` key가 없거나 문자열로 남아 있으면 뒤 문제는 모두 흔들린다.

In [ ]:
# 제출 전 선택 점검 셀: 필요할 때만 실행하세요.
print('products type:', type(products))
print('products count:', len(products))
print('first product:', products[0] if products else '비어 있음')
print('notices count:', len(notices) if 'notices' in globals() else 'notices 없음')


## 좋은 제출물 기준

좋은 제출물은 정답 숫자만 맞는 노트북이 아니다. 다른 사람이 열어도 "어떤 HTML에서 무엇을 뽑았고, 어떤 기준으로 필터링했는지"를 따라갈 수 있어야 한다.

- 변수명은 `a`, `b`, `x` 보다 `products`, `notices`, `sold_out` 처럼 의미 있게 쓴다.
- 중간 결과는 너무 많이 출력하지 말고, 개수와 첫 번째 샘플 정도만 확인한다.
- 마지막 요약 문장은 출력된 숫자와 맞아야 한다.
- 실패했던 임시 코드는 제출 전에 정리한다.
- 실제 사이트에 적용하겠다는 문장을 쓸 때는 robots.txt, 약관, 요청 간격 확인을 함께 적는다.

## 다음 레슨 준비 질문

2강에서는 한 페이지가 아니라 여러 URL을 순서대로 읽는다. 오늘 만든 코드에서 다음 질문에 답할 수 있으면 2강 준비가 된 것이다.

1. 파일 이름만 바뀌어도 `load_text()` 로 같은 방식으로 읽을 수 있는가?
2. 반복 단위 selector가 페이지마다 같다면 같은 함수를 재사용할 수 있는가?
3. 상대 URL을 절대 URL로 바꾸지 않으면 여러 페이지 결과를 합칠 때 어떤 문제가 생기는가?
4. 요청 간격을 어디에 넣어야 여러 페이지 처리에서도 안전한가?

# 레슨 01 — 최종 미션: 샘플 쇼핑/공지 페이지 수집 리포트

이번 최종 미션은 mini_shop.html과 notices.html을 하나의 작은 수집 리포트로 정리하는 것이다. 실제 사이트가 아니라 수업용 합성 HTML이므로 요청 제한 문제 없이 반복 연습할 수 있다.

## 목표

1. 상품 카드 전체를 파싱해 products 리스트를 만든다.
2. 상품 결과를 lesson01_products.csv로 저장한다.
3. 공지 목록 전체를 파싱해 notices 리스트를 만든다.
4. 품절 상품, 고평점 상품, 조회수 상위 공지를 요약한다.
5. 마지막 마크다운 셀에 자동화 윤리 체크리스트를 작성한다.

## 환경 셀

In [ ]:
import os
import re
import time
import csv
from pathlib import Path
from urllib.parse import urljoin

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/01/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', text))

print('colab:', IS_COLAB)
print('data base:', DATA_BASE)


## 진행 셀

In [ ]:
# 1. HTML 읽기
shop_html = load_text('mini_shop.html')
notice_html = load_text('notices.html')

# 2. BeautifulSoup 객체 만들기
shop_soup = BeautifulSoup(____, 'html.parser')
notice_soup = BeautifulSoup(____, 'html.parser')

# 3. 상품 추출
products = []
for card in shop_soup.select('____'):
    products.append({
        'name': ____,
        'category': ____,
        'price': ____,
        'rating': ____,
        'stock': ____,
        'detail_url': ____,
    })

# 4. 공지 추출
notices = []
for row in notice_soup.select('____'):
    notices.append({
        'date': ____,
        'title': ____,
        'department': ____,
        'views': ____,
    })

print(len(products), len(notices))


## CSV 저장 셀

In [ ]:
# products를 CSV로 저장하세요.
with open('lesson01_products.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['name', 'category', 'price', 'rating', 'stock', 'detail_url'])
    writer.writeheader()
    writer.writerows(____)
print('saved')


## 요약 셀

In [ ]:
# 품절 상품, 고평점 상품, 조회수 상위 공지를 요약하세요.
sold_out = ____
top_rated = ____
top_notices = ____

print('품절 상품:', ____)
print('고평점 상품:', ____)
print('조회수 상위 공지:', ____)


## 결론 작성

아래 항목을 짧게 작성한다.

- 수집한 상품 수 / 공지 수
- 운영자가 먼저 봐야 할 품절 또는 고평점 상품
- 조회수 상위 공지가 의미하는 것
- 실제 사이트에서 같은 코드를 실행하기 전에 확인해야 할 윤리/안전 항목 3개

## 필수 요구사항

최종 미션은 아래 5개를 모두 만족해야 완료로 본다.

1. `mini_shop.html` 의 모든 상품 카드를 빠짐없이 읽어 `products` 리스트를 만든다.
2. `notices.html` 의 모든 공지 행을 빠짐없이 읽어 `notices` 리스트를 만든다.
3. 상품 결과를 `lesson01_products.csv` 로 저장하고, 헤더가 포함되어야 한다.
4. 품절 상품, 평점 4.7 이상 상품, 조회수 상위 공지 3개를 각각 출력한다.
5. 마지막 마크다운 셀에 실제 사이트 적용 전 안전 체크 3가지를 적는다.

## 보너스 요구사항

시간이 남는 학생은 아래 중 1개 이상을 추가한다.

- 카테고리별 상품 수와 평균 가격을 계산한다.
- 부서별 공지 수와 평균 조회수를 계산한다.
- `targets.csv` 에서 allowed가 `yes` 인 항목만 읽어 처리 대상 목록을 만든다.
- 결과 CSV 파일명을 날짜가 포함된 형태로 바꾼다. 예: `lesson01_products_2026-05-17.csv`

## 제출 전 검증

제출 직전에는 런타임을 다시 시작한 뒤 위에서부터 다시 실행한다. 중간 셀부터 실행해야만 동작하는 노트북은 완료로 보지 않는다.

In [ ]:
# 선택 검증 셀
print('상품 수:', len(products))
print('공지 수:', len(notices))
print('CSV 저장 대상 행 수:', len(products))
print('품절 상품 수:', len(sold_out))
print('고평점 상품 수:', len(top_rated))
print('조회수 상위 공지 수:', len(top_notices))


## 평가 루브릭

| 항목 | 기준 | 점수 |
|---|---|---:|
| 데이터 로드 | HTML 두 개를 모두 읽고 BeautifulSoup 객체로 변환 | 2 |
| 상품 파싱 | name/category/price/rating/stock/detail_url 추출 | 2 |
| 공지 파싱 | date/title/department/views 추출 | 2 |
| 저장 | CSV 헤더와 행이 정상 저장 | 1 |
| 요약 | 품절/고평점/조회수 상위 결과가 문장으로 정리 | 2 |
| 안전 메모 | robots.txt, 요청 간격, 개인정보/약관 확인 언급 | 1 |

8점 이상이면 완료, 10점이면 우수로 본다.